# Andes Capital Real Estate
## 03. Business Performance Analysis

### Business Problem

Andes Capital needs a consolidated view of its real estate sales performance to understand revenue trends, property demand, sales channels, and customer purchasing behavior.

### Business Objective

Evaluate commercial performance across 2023–2024 and identify opportunities to improve sales strategy and business decision-making.

### Key Business Questions

1. How did sales value and commission revenue evolve between 2023 and 2024?
2. Which property types and cities contribute most to sales performance?
3. How does commercial performance differ between sales channels?
4. Which customer segments generate the highest sales value, and how frequently do customers return?

### Analytical Approach

- Define and validate business KPIs.
- Analyze commercial performance and customer behavior.
- Identify relevant patterns and differences.
- Translate findings into actionable business recommendations.
- Build an executive Power BI dashboard.

In [1]:
# Import libraries
import pandas as pd
from pathlib import Path

# Define project directory
project_dir = Path.cwd()

if not (project_dir / "data").exists():
    project_dir = project_dir.parent

processed_dir = project_dir / "data" / "processed"

# Load prepared datasets
sales = pd.read_csv(
    processed_dir / "fact_sales.csv",
    parse_dates=["fecha_venta"]
)

customers = pd.read_csv(
    processed_dir / "dim_customers.csv"
)

properties = pd.read_csv(
    processed_dir / "dim_properties.csv"
)

print("Prepared datasets loaded successfully!")

Prepared datasets loaded successfully!


In [2]:
# Calculate key business performance indicators

total_sales = sales["precio_venta"].sum()

commission_revenue = sales["monto_comision"].sum()

total_transactions = sales["id_venta"].nunique()

average_sale_value = sales["precio_venta"].mean()

unique_buyers = sales["id_cliente"].nunique()

# Display results
print(f"Total Sales Value: {total_sales:,.2f}")
print(f"Commission Revenue: {commission_revenue:,.2f}")
print(f"Total Transactions: {total_transactions:,}")
print(f"Average Sale Value: {average_sale_value:,.2f}")
print(f"Unique Buyers: {unique_buyers:,}")

Total Sales Value: 6,012,502,170.00
Commission Revenue: 200,627,166.00
Total Transactions: 8,500
Average Sale Value: 707,353.20
Unique Buyers: 3,139


In [3]:
# Analyze annual business performance

annual_performance = (
    sales
    .groupby(sales["fecha_venta"].dt.year)
    .agg(
        total_sales=("precio_venta", "sum"),
        commission_revenue=("monto_comision", "sum"),
        transactions=("id_venta", "nunique"),
        unique_buyers=("id_cliente", "nunique")
    )
)

display(annual_performance)

,total_sales,commission_revenue,transactions,unique_buyers
fecha_venta,,,,
2023,2847702456,94735225,4045,2366
2024,3164799714,105891941,4455,2484


In [4]:
# Calculate year-over-year growth

annual_growth = annual_performance.pct_change() * 100

display(annual_growth.round(2))

,total_sales,commission_revenue,transactions,unique_buyers
fecha_venta,,,,
2023,NaN,NaN,NaN,NaN
2024,11.14,11.78,10.14,4.99


In [5]:
# Analyze average sale value by year

annual_performance["average_sale_value"] = (
    annual_performance["total_sales"]
    / annual_performance["transactions"]
)

average_sale_growth = (
    annual_performance["average_sale_value"]
    .pct_change() * 100
)

display(
    annual_performance[
        ["total_sales", "transactions", "average_sale_value"]
    ].round(2)
)

print(
    f"\nAverage Sale Value YoY Growth: "
    f"{average_sale_growth.loc[2024]:.2f}%"
)

,total_sales,transactions,average_sale_value
fecha_venta,,,
2023,2847702456,4045,704005.55
2024,3164799714,4455,710392.75



Average Sale Value YoY Growth: 0.91%


### Finding 1 — Sales Growth Driven by Transaction Volume

**Context**

Andes Capital increased its total sales value by 11.14% between 2023 and 2024.

**Finding**

The number of transactions increased by 10.14%, while the average sale value grew by only 0.91%.

This indicates that sales growth was primarily driven by a higher number of transactions rather than a substantial increase in average transaction value.

**Business Implication**

Further analysis should identify which property types, cities, and sales channels contributed most to transaction growth.

Understanding these differences may help Andes Capital prioritize commercial opportunities and allocate resources more effectively.

In [6]:
# Analyze monthly business performance

monthly_performance = (
    sales
    .groupby(sales["fecha_venta"].dt.to_period("M"))
    .agg(
        total_sales=("precio_venta", "sum"),
        commission_revenue=("monto_comision", "sum"),
        transactions=("id_venta", "nunique")
    )
)

display(monthly_performance)

,total_sales,commission_revenue,transactions
fecha_venta,,,
2023-01,162741008,5383002,224
2023-02,180172127,5716076,247
2023-03,346782891,11715435,434
2023-04,330950703,10747819,473
2023-05,186047939,6267890,290
2023-06,186643186,6001942,270
2023-07,163044045,5504314,241
2023-08,166773927,5656038,268
2023-09,294152624,10044108,405


In [7]:
# Calculate monthly year-over-year growth

monthly_yoy = (
    monthly_performance[
        ["total_sales", "commission_revenue", "transactions"]
    ]
    .pct_change(periods=12) * 100
)

display(monthly_yoy.loc["2024"].round(2))

,total_sales,commission_revenue,transactions
fecha_venta,,,
2024-01,9.67,15.18,19.20
2024-02,16.97,26.49,12.15
2024-03,1.41,1.33,14.52
2024-04,8.14,7.97,4.44
2024-05,25.40,22.78,15.52
2024-06,10.15,17.20,12.59
2024-07,27.22,26.04,17.43
2024-08,20.59,24.74,1.12
2024-09,9.33,6.86,8.89


### Finding 2 — Broad-Based Growth with Monthly Variations

**Context**

Andes Capital recorded an 11.14% increase in total sales value in 2024 compared with 2023.

**Finding**

Sales value increased year-over-year in 11 of 12 months.

July (+27.22%) and May (+25.40%) recorded the strongest growth, while October was the only month with a slight decline (-0.84%).

In August, sales value increased by 20.59%, while transactions grew by only 1.12%, suggesting a higher average transaction value.

**Business Implication**

The company experienced broad-based growth, although monthly performance varied.

Further analysis by property type, city, and sales channel is needed to understand which segments contributed to these differences.

In [8]:
# Analyze business performance by city

sales_by_city = sales.merge(
    properties[["id_propiedad", "ciudad"]],
    on="id_propiedad",
    how="left",
    validate="many_to_one"
)

city_performance = (
    sales_by_city
    .groupby("ciudad")
    .agg(
        total_sales=("precio_venta", "sum"),
        commission_revenue=("monto_comision", "sum"),
        transactions=("id_venta", "nunique")
    )
    .sort_values("total_sales", ascending=False)
)

display(city_performance)

,total_sales,commission_revenue,transactions
ciudad,,,
Ciudad de México,3242231285,108204490,4123
Bogotá,2770270885,92422676,4377


In [9]:
# Analyze sales performance by city and property type

property_performance = (
    sales_by_city
    .merge(
        properties[["id_propiedad", "tipo_propiedad"]],
        on="id_propiedad",
        how="left",
        validate="many_to_one"
    )
    .groupby(["ciudad", "tipo_propiedad"])
    .agg(
        total_sales=("precio_venta", "sum"),
        transactions=("id_venta", "nunique"),
        average_sale_value=("precio_venta", "mean")
    )
    .sort_values("total_sales", ascending=False)
)

display(property_performance.round(2))

total_sales  transactions  average_sale_value
ciudad           tipo_propiedad                                               
Ciudad de México Casa             1205259422          1117          1079014.70
                 Comercial        1063458181           532          1998981.54
Bogotá           Casa             1035275882          1207           857726.50
Ciudad de México Departamento      973513682          2474           393497.85
Bogotá           Departamento      873761550          2631           332102.45
                 Comercial         861233453           539          1597835.72


### Finding 3 — Higher Transaction Values in Mexico City

**Context**

Mexico City generates more total sales value than Bogotá despite recording fewer transactions.

**Finding**

Average sale values are higher in Mexico City across all three property types:

- Houses: +25.8%
- Commercial properties: +25.1%
- Apartments: +18.5%

Apartments account for approximately 60% of transactions in both cities, indicating a similar transaction mix by property type.

**Business Implication**

The difference in commercial performance is not explained solely by property type distribution.

Further analysis should examine differences in property characteristics, transaction values, and commercial channels before recommending market-specific strategies.


In [10]:
# Analyze performance by sales channel

channel_performance = (
    sales
    .groupby("canal_venta")
    .agg(
        total_sales=("precio_venta", "sum"),
        commission_revenue=("monto_comision", "sum"),
        transactions=("id_venta", "nunique"),
        average_sale_value=("precio_venta", "mean")
    )
    .sort_values("total_sales", ascending=False)
)

display(channel_performance.round(2))

,total_sales,commission_revenue,transactions,average_sale_value
canal_venta,,,,
Corredor,4379990279,176173560,6181,708621.63
Directo,1632511891,24453606,2319,703972.35


In [11]:
# Analyze effective commission rates by sales channel

channel_performance["effective_commission_rate"] = (
    channel_performance["commission_revenue"]
    / channel_performance["total_sales"]
) * 100

display(
    channel_performance[
        ["commission_revenue", "total_sales",
         "effective_commission_rate"]
    ].round(2)
)

,commission_revenue,total_sales,effective_commission_rate
canal_venta,,,
Corredor,176173560,4379990279,4.02
Directo,24453606,1632511891,1.50



### Finding 4 — Commission Revenue Concentration by Sales Channel

**Context**

Andes Capital operates through two sales channels: Broker (Corredor) and Direct Sales (Directo).

**Finding**

The Broker channel accounts for approximately 72.85% of total sales value but generates 87.81% of recorded commissions.

Its effective commission rate is 4.02%, compared with 1.50% for Direct Sales.

Average transaction values are similar across both channels, indicating that differences in commission rates and transaction volume explain most of the commission revenue gap.

**Business Implication**

Andes Capital should evaluate the commission structure and associated operating costs of each channel before making resource allocation decisions.

Higher commission revenue does not necessarily imply higher profitability.


In [12]:
# Analyze sales performance by customer segment

sales_customers = sales.merge(
    customers[["id_cliente", "segmento_comprador"]],
    on="id_cliente",
    how="left",
    validate="many_to_one"
)

segment_performance = (
    sales_customers
    .groupby("segmento_comprador")
    .agg(
        total_sales=("precio_venta", "sum"),
        commission_revenue=("monto_comision", "sum"),
        transactions=("id_venta", "nunique"),
        unique_buyers=("id_cliente", "nunique"),
        average_sale_value=("precio_venta", "mean")
    )
    .sort_values("total_sales", ascending=False)
)

display(segment_performance.round(2))

,total_sales,commission_revenue,transactions,unique_buyers,average_sale_value
segmento_comprador,,,,,
Primera vez,3783579998,126126336,5302,1770,713613.73
Inversionista,1471444845,48866744,2093,914,703031.46
Alto patrimonio,757477327,25634086,1105,455,685499.84


In [13]:
# Analyze purchase frequency by customer

purchase_frequency = (
    sales_customers
    .groupby(["id_cliente", "segmento_comprador"])
    .agg(
        transactions=("id_venta", "nunique"),
        total_sales=("precio_venta", "sum")
    )
    .reset_index()
)

frequency_by_segment = (
    purchase_frequency
    .groupby("segmento_comprador")
    .agg(
        unique_buyers=("id_cliente", "nunique"),
        average_transactions=("transactions", "mean"),
        repeat_buyers=("transactions", lambda x: (x > 1).sum())
    )
)

frequency_by_segment["repeat_buyer_rate"] = (
    frequency_by_segment["repeat_buyers"]
    / frequency_by_segment["unique_buyers"]
) * 100

display(frequency_by_segment.round(2))

,unique_buyers,average_transactions,repeat_buyers,repeat_buyer_rate
segmento_comprador,,,,
Alto patrimonio,455,2.43,339,74.51
Inversionista,914,2.29,628,68.71
Primera vez,1770,3.00,1454,82.15


In [14]:
# Analyze overall purchase frequency distribution

frequency_distribution = (
    purchase_frequency["transactions"]
    .value_counts()
    .sort_index()
)

display(frequency_distribution)

transactions
1     718
2     913
3     704
4     428
5     216
6      97
7      42
8      16
9       2
10      3
Name: count, dtype: int64

In [15]:
# Validate distinct properties purchased by each customer

customer_properties = (
    sales
    .groupby("id_cliente")
    .agg(
        transactions=("id_venta", "nunique"),
        unique_properties=("id_propiedad", "nunique")
    )
)

print(
    "Customers with repeated transactions for the same property:",
    (customer_properties["transactions"] >
     customer_properties["unique_properties"]).sum()
)


Customers with repeated transactions for the same property: 1


In [16]:
# Identify customers with repeated property transactions

repeated_properties = customer_properties[
    customer_properties["transactions"] >
    customer_properties["unique_properties"]
]

display(repeated_properties)

,transactions,unique_properties
id_cliente,,
CUST00220,3,2


In [17]:
# Inspect repeated property transactions

customer_transactions = sales[
    sales["id_cliente"] == "CUST00220"
].sort_values("id_propiedad")

display(customer_transactions)

,id_venta,fecha_venta,id_cliente,id_propiedad,precio_venta,canal_venta,porcentaje_comision,monto_comision
2733,SALE002734,2024-03-07,CUST00220,PROP02560,1902355,Directo,0.0147,28020
7056,SALE007057,2024-12-15,CUST00220,PROP02560,1877453,Corredor,0.0410,76888
1437,SALE001438,2024-04-25,CUST00220,PROP07408,399571,Corredor,0.0346,13811



### Data Quality Note — Repeated Property Transactions

A customer-level validation identified one buyer (CUST00220) with three transactions involving two distinct properties.

Property PROP02560 appears in two transactions with different dates, sale prices, and sales channels.

These records are not exact duplicates. However, the available data does not establish whether they represent separate legitimate transactions or a recording inconsistency.

**Decision:** Retain all transactions and document the anomaly.

Purchase frequency and repeat buyer metrics will be based on distinct transaction IDs, not distinct properties.


In [18]:
# Identify the first recorded purchase of each customer

first_purchase = (
    sales
    .groupby("id_cliente")["fecha_venta"]
    .min()
    .reset_index(name="first_purchase_date")
)

first_purchase["cohort_month"] = (
    first_purchase["first_purchase_date"]
    .dt.to_period("M")
)

display(first_purchase.head())

,id_cliente,first_purchase_date,cohort_month
0,CUST00001,2023-12-04,2023-12
1,CUST00003,2023-03-23,2023-03
2,CUST00004,2023-04-27,2023-04
3,CUST00005,2023-04-22,2023-04
4,CUST00006,2023-04-30,2023-04


In [19]:
# Assign each transaction to its customer's cohort

cohort_data = sales.merge(
    first_purchase[["id_cliente", "cohort_month"]],
    on="id_cliente",
    how="left",
    validate="many_to_one"
)

# Identify the month of each transaction
cohort_data["purchase_month"] = (
    cohort_data["fecha_venta"].dt.to_period("M")
)

display(
    cohort_data[
        ["id_cliente", "fecha_venta", "cohort_month", "purchase_month"]
    ].head()
)

,id_cliente,fecha_venta,cohort_month,purchase_month
0,CUST02497,2024-01-05,2024-01,2024-01
1,CUST02903,2024-06-16,2024-03,2024-06
2,CUST00831,2024-10-13,2024-10,2024-10
3,CUST01686,2023-02-08,2023-01,2023-02
4,CUST00639,2024-11-29,2024-07,2024-11


In [20]:
# Calculate months since first recorded purchase

cohort_data["cohort_index"] = (
    (cohort_data["purchase_month"].dt.year -
     cohort_data["cohort_month"].dt.year) * 12
    +
    (cohort_data["purchase_month"].dt.month -
     cohort_data["cohort_month"].dt.month)
)

display(
    cohort_data[
        ["id_cliente", "cohort_month",
         "purchase_month", "cohort_index"]
    ].head()
)

,id_cliente,cohort_month,purchase_month,cohort_index
0,CUST02497,2024-01,2024-01,0
1,CUST02903,2024-03,2024-06,3
2,CUST00831,2024-10,2024-10,0
3,CUST01686,2023-01,2023-02,1
4,CUST00639,2024-07,2024-11,4


In [21]:
# Calculate unique buyers by cohort and month

cohort_counts = (
    cohort_data
    .groupby(["cohort_month", "cohort_index"])["id_cliente"]
    .nunique()
    .unstack()
)

display(cohort_counts.head())

cohort_index,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
cohort_month,,,,,,,,,,,,,,,,,,,,,
2023-01,212.0,11.0,32.0,29.0,20.0,14.0,16.0,14.0,22.0,31.0,...,25.0,27.0,19.0,20.0,18.0,10.0,20.0,26.0,23.0,23.0
2023-02,227.0,26.0,29.0,18.0,14.0,22.0,14.0,30.0,25.0,35.0,...,36.0,19.0,27.0,22.0,22.0,32.0,34.0,24.0,18.0,NaN
2023-03,353.0,47.0,34.0,24.0,23.0,28.0,42.0,41.0,57.0,27.0,...,33.0,27.0,29.0,28.0,35.0,50.0,66.0,31.0,NaN,NaN
2023-04,346.0,20.0,28.0,21.0,24.0,37.0,44.0,42.0,41.0,24.0,...,26.0,26.0,29.0,42.0,49.0,42.0,33.0,NaN,NaN,NaN
2023-05,184.0,16.0,10.0,15.0,23.0,29.0,25.0,15.0,14.0,8.0,...,14.0,8.0,23.0,23.0,22.0,25.0,NaN,NaN,NaN,NaN


In [22]:
# Calculate monthly cohort repurchase rates

cohort_sizes = cohort_counts[0]

cohort_rates = (
    cohort_counts
    .div(cohort_sizes, axis=0) * 100
)

display(cohort_rates.iloc[:5, :7].round(2))

cohort_index,0,1,2,3,4,5,6
cohort_month,,,,,,,
2023-01,100.0,5.19,15.09,13.68,9.43,6.60,7.55
2023-02,100.0,11.45,12.78,7.93,6.17,9.69,6.17
2023-03,100.0,13.31,9.63,6.80,6.52,7.93,11.90
2023-04,100.0,5.78,8.09,6.07,6.94,10.69,12.72
2023-05,100.0,8.70,5.43,8.15,12.50,15.76,13.59


In [23]:
# Calculate weighted cohort repurchase rates

for month in [1, 2, 3, 6]:
    eligible = cohort_counts[month].notna()

    active_buyers = cohort_counts.loc[eligible, month].sum()
    initial_buyers = cohort_sizes.loc[eligible].sum()

    rate = active_buyers / initial_buyers * 100

    print(f"Month {month} repurchase rate: {rate:.2f}%")

Month 1 repurchase rate: 9.35%
Month 2 repurchase rate: 9.77%
Month 3 repurchase rate: 9.47%
Month 6 repurchase rate: 10.21%


In [24]:
# Correct cohort matrix: distinguish zero activity from unobserved months

cohort_counts_corrected = cohort_counts.copy()

last_month = sales["fecha_venta"].max().to_period("M")

for cohort in cohort_counts_corrected.index:
    max_observed_month = (
        (last_month.year - cohort.year) * 12
        + (last_month.month - cohort.month)
    )

    valid_columns = [
        month for month in cohort_counts_corrected.columns
        if month <= max_observed_month
    ]

    cohort_counts_corrected.loc[
        cohort, valid_columns
    ] = cohort_counts_corrected.loc[
        cohort, valid_columns
    ].fillna(0)

display(cohort_counts_corrected.iloc[:5, :7])

cohort_index,0,1,2,3,4,5,6
cohort_month,,,,,,,
2023-01,212.0,11.0,32.0,29.0,20.0,14.0,16.0
2023-02,227.0,26.0,29.0,18.0,14.0,22.0,14.0
2023-03,353.0,47.0,34.0,24.0,23.0,28.0,42.0
2023-04,346.0,20.0,28.0,21.0,24.0,37.0,44.0
2023-05,184.0,16.0,10.0,15.0,23.0,29.0,25.0


In [25]:
# Recalculate weighted cohort repurchase rates

cohort_sizes = cohort_counts_corrected[0]

for month in [1, 2, 3, 6]:
    eligible = cohort_counts_corrected[month].notna()

    active_buyers = cohort_counts_corrected.loc[eligible, month].sum()
    initial_buyers = cohort_sizes.loc[eligible].sum()

    rate = active_buyers / initial_buyers * 100

    print(f"Month {month} repurchase rate: {rate:.2f}%")

Month 1 repurchase rate: 9.35%
Month 2 repurchase rate: 9.77%
Month 3 repurchase rate: 9.47%
Month 6 repurchase rate: 10.21%



### Finding 5 — Monthly Repurchase Behavior

**Context**

Andes Capital recorded 3,139 unique buyers between January 2023 and December 2024.

Of these, 77.13% completed more than one transaction during the observed period.

**Finding**

Cohort analysis shows the following weighted monthly repurchase rates:

- Month 1: 9.35%
- Month 2: 9.77%
- Month 3: 9.47%
- Month 6: 10.21%

These results suggest relatively stable repurchase activity across the observed cohort months.

**Business Implication**

The company should investigate which customer segments contribute most to repeat purchasing and whether targeted relationship management could support additional transactions.

**Limitations**

Monthly repurchase rates measure purchasing activity in specific months, not continuous customer retention.

Only cohorts with sufficient observation time were included in each calculation. The dataset does not contain transactions before January 2023.



In [26]:
# Build final cohort repurchase matrix

cohort_rates_final = (
    cohort_counts_corrected
    .div(cohort_sizes, axis=0) * 100
)

display(cohort_rates_final.iloc[:10, :7].round(2))

cohort_index,0,1,2,3,4,5,6
cohort_month,,,,,,,
2023-01,100.0,5.19,15.09,13.68,9.43,6.60,7.55
2023-02,100.0,11.45,12.78,7.93,6.17,9.69,6.17
2023-03,100.0,13.31,9.63,6.80,6.52,7.93,11.90
2023-04,100.0,5.78,8.09,6.07,6.94,10.69,12.72
2023-05,100.0,8.70,5.43,8.15,12.50,15.76,13.59
2023-06,100.0,7.74,4.52,12.26,12.90,10.32,7.10
2023-07,100.0,4.69,10.94,8.59,14.84,6.25,8.59
2023-08,100.0,8.28,10.34,10.34,7.59,8.97,10.34
2023-09,100.0,8.89,17.22,8.89,9.44,8.89,13.33


In [27]:
# Validate final cohort rates

print(
    "Cohorts with initial rate different from 100%:",
    (cohort_rates_final[0] != 100).sum()
)

print(
    "Rates below 0%:",
    (cohort_rates_final < 0).sum().sum()
)

print(
    "Rates above 100%:",
    (cohort_rates_final > 100).sum().sum()
)

Cohorts with initial rate different from 100%: 0
Rates below 0%: 0
Rates above 100%: 0


In [28]:
# Prepare cohort analysis for Power BI

cohort_export = (
    cohort_rates_final
    .stack()
    .reset_index()
)

cohort_export.columns = [
    "cohort_month",
    "cohort_index",
    "repurchase_rate"
]

cohort_export["cohort_month"] = (
    cohort_export["cohort_month"].astype(str)
)

display(cohort_export.head(10))

,cohort_month,cohort_index,repurchase_rate
0,2023-01,0,100.000000
1,2023-01,1,5.188679
2,2023-01,2,15.094340
3,2023-01,3,13.679245
4,2023-01,4,9.433962
5,2023-01,5,6.603774
6,2023-01,6,7.547170
7,2023-01,7,6.603774
8,2023-01,8,10.377358
9,2023-01,9,14.622642


In [29]:
# Export cohort analysis results

cohort_export.to_csv(
    processed_dir / "cohort_repurchase_rates.csv",
    index=False
)

print("Cohort analysis exported successfully!")

Cohort analysis exported successfully!


In [30]:
# Final reconciliation of business metrics

print(f"Transactions: {sales['id_venta'].nunique():,}")
print(f"Total Sales Value: {sales['precio_venta'].sum():,.2f}")
print(f"Commission Revenue: {sales['monto_comision'].sum():,.2f}")
print(f"Unique Buyers: {sales['id_cliente'].nunique():,}")
print(f"Cohort Buyers: {cohort_sizes.sum():,}")

Transactions: 8,500
Total Sales Value: 6,012,502,170.00
Commission Revenue: 200,627,166.00
Unique Buyers: 3,139
Cohort Buyers: 3,139.0
